# Piloto EFT-SP — VMamba Tiny

200 atualizações, full fine-tuning, AdamW lr=1e-5, sem weight decay, batch 1, float32, CE + alinhamento. Último estágio com normalização pré-treinada, entrada 256 e grade ALP 32×32.

Este piloto usa **200 imagens distintas** da divisão de treino, uma região escolhida por imagem. A validação usa uma região fixa por imagem elegível reservada, antes e depois. Aumentações limitadas ao flip horizontal da mesma imagem/região, como no diagnóstico. Não é treino completo nem reprodução exata da receita do artigo.

Na validação, `eval()` desativa alinhamento e drop path. `isval=False` mantém o kernel ALP 4×4. Registramos CE e Dice de foreground de superpixels: não são métricas de pólipos. Não selecionamos checkpoints pelas imagens de teste.

Execute uma variante por vez. Comece pelo Tiny; envie os resultados antes de iniciar Small/Base.

**O piloto mantém os pesos adaptados em `checkpoint_latest.pt`.** Os checkpoints pré-treinados de origem nunca são alterados. A gravação ocorre antes do treino e a cada 25 atualizações; não há uma cópia por passo. A retomada refaz somente passos posteriores à última gravação.


In [ ]:
from pathlib import Path
import sys

ESPERADO = "/workspace/miniforge3/envs/protosam/bin/python"
print("Python do kernel:", sys.executable)
assert Path(sys.executable).absolute() == Path(ESPERADO), "Selecione Kernel → Change Kernel → Python (protosam)."
REPO = Path("/workspace/ProtoSAM-repro")
PASTA = REPO / "exp/eft_sp_piloto_v1"
assert (PASTA / "eft_dados.py").is_file(), PASTA
for caminho in (REPO, PASTA):
    if str(caminho) not in sys.path:
        sys.path.insert(0, str(caminho))


In [ ]:
from pathlib import Path
from eft_treino import executar_piloto

PASTA_DADOS = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "eft_sp_piloto_dados_20261011T152315570066Z"
)
assert (PASTA_DADOS / "planos.json").is_file()

VARIANTE = "tiny"
RETOMAR = None

In [ ]:
PASTA_EXECUCAO = executar_piloto(
    dados=PASTA_DADOS,
    variante=VARIANTE,
    retomar=RETOMAR,
    repo=REPO,
)
print("RESULTADOS:", PASTA_EXECUCAO)

Envie o resumo final. Se houver erro, envie o traceback completo e o caminho EXECUÇÃO. Não mude filtros, loss ou taxa de aprendizado para contornar uma falha sem registrá-la. Em uma retomada, logs são separados por tentativa e indicam o passo inicial. Mesmo com RNG salvo, determinismo bit a bit dos kernels CUDA não foi comprovado.

In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
from PIL import Image, ImageDraw
from IPython.display import display

from eft_modelo import construir_modelo
from eft_dados import carregar_episodio, agora

EXECUCAO = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "eft_tiny_piloto_20261011T152746615659Z"
)
config = json.loads((EXECUCAO / "config.json").read_text())
DADOS = Path(config["dados"])
planos = json.loads((DADOS / "planos.json").read_text())
validacao = planos["validacao"]

# Quatro posições distribuídas pela lista, independentes do Dice.
indices = np.linspace(0, len(validacao) - 1, 4, dtype=int)
selecionados = [validacao[int(i)] for i in indices]

antes = json.loads(
    (EXECUCAO / "validacao_inicial.json").read_text()
)
depois = json.loads(
    (EXECUCAO / "validacao_final.json").read_text()
)
metricas_antes = {r["indice"]: r for r in antes["por_imagem"]}
metricas_depois = {r["indice"]: r for r in depois["por_imagem"]}

# Resumo pareado de todos os episódios, não apenas dos quatro exibidos.
deltas = np.array([
    metricas_depois[i]["dice_pseudo_fg"]
    - metricas_antes[i]["dice_pseudo_fg"]
    for i in metricas_antes
])
print(
    f"Dos {len(deltas)} episódios: "
    f"{(deltas > 1e-8).sum()} melhoraram, "
    f"{(deltas < -1e-8).sum()} pioraram, "
    f"{(np.abs(deltas) <= 1e-8).sum()} empataram."
)
print(f"ΔDice mediano: {np.median(deltas):+.6f}")


def prever(modelo, registros):
    saidas = []
    modelo.eval()
    with torch.no_grad():
        for registro in registros:
            support, fg, query, alvo = carregar_episodio(
                DADOS, registro, "cuda"
            )
            retorno = modelo(
                [[support]], [[fg]], [[1 - fg]], [query],
                isval=False, val_wsize=2, show_viz=False,
            )
            logits = retorno[0]
            assert torch.isfinite(logits).all().item()
            saidas.append(
                logits.argmax(1)[0].cpu().numpy().astype(np.uint8)
            )
            del retorno, logits
    return saidas


# Um único modelo na GPU: primeiro pesos originais, depois adaptados.
modelo_visual = construir_modelo("tiny")

try:
    predicoes_antes = prever(modelo_visual, selecionados)

    checkpoint = torch.load(
        EXECUCAO / "checkpoint_latest.pt",
        map_location="cpu",
        weights_only=True,
    )
    assert checkpoint["passo"] == 200
    modelo_visual.load_state_dict(checkpoint["modelo"], strict=True)
    del checkpoint

    predicoes_depois = prever(modelo_visual, selecionados)
finally:
    del modelo_visual
    torch.cuda.empty_cache()


def imagem_mascara(mascara):
    return Image.fromarray(
        mascara.astype(np.uint8) * 255
    ).convert("RGB")


altura_linha = 310
painel = Image.new("RGB", (4 * 256, 4 * altura_linha), "white")
desenho = ImageDraw.Draw(painel)

for linha, registro in enumerate(selecionados):
    with np.load(DADOS / registro["cache"], allow_pickle=False) as dados:
        query_rgb = np.fliplr(dados["rgb"]).copy()
        alvo = np.fliplr(
            (dados["segmentos"] == registro["regiao"]) & dados["util"]
        ).copy()

    identificador = registro["indice"]
    dice_a = metricas_antes[identificador]["dice_pseudo_fg"]
    dice_d = metricas_depois[identificador]["dice_pseudo_fg"]

    imagens = [
        Image.fromarray(query_rgb),
        imagem_mascara(alvo),
        imagem_mascara(predicoes_antes[linha]),
        imagem_mascara(predicoes_depois[linha]),
    ]
    titulos = [
        "Query",
        "Alvo: superpixel",
        f"Antes | Dice={dice_a:.3f}",
        f"Depois | Dice={dice_d:.3f}",
    ]

    y = linha * altura_linha
    desenho.text(
        (5, y + 4),
        f"{registro['nome']} | regiao {registro['regiao']}",
        fill="black",
    )
    for coluna, (imagem, titulo) in enumerate(zip(imagens, titulos)):
        x = coluna * 256
        desenho.text((x + 5, y + 24), titulo, fill="black")
        painel.paste(imagem, (x, y + 48))

destino = EXECUCAO / ("inspecao_visual_" + agora())
destino.mkdir()
painel.save(destino / "antes_depois.png")
(destino / "selecao.json").write_text(
    json.dumps(
        {
            "criterio": "quatro posições igualmente espaçadas na validação",
            "indices_na_lista": indices.tolist(),
            "episodios": selecionados,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

display(painel)
print("Painel salvo em:", destino / "antes_depois.png")